# YouTube Content Performance & Engagement Analysis

## Data Collection, Cleaning & Feature Engineering

This notebook documents the Python workflow used in the **YouTube Content Performance & Engagement Analysis** project.

### Workflow
1. Connect to the YouTube Data API
2. Identify the target channel and uploads playlist
3. Collect video IDs using API pagination
4. Retrieve video metadata and statistics
5. Clean and standardize the dataset
6. Create engagement, date, and duration features
7. Perform exploratory analysis
8. Export the SQL-ready dataset for MySQL and Power BI

**Data source:** YouTube Data API  
**Target channel used in the original project:** Apna College Official

> **Security:** The API key from the original notebook has been removed. For GitHub, set it locally as the `YOUTUBE_API_KEY` environment variable.


## 1. Import Libraries

In [ ]:
import os
import re
import pandas as pd
from googleapiclient.discovery import build


## 2. Configure the YouTube API

Set `YOUTUBE_API_KEY` in your local environment before running this notebook.
Never hard-code an API key in a public GitHub repository.


In [ ]:
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY is not set. "
        "Create an environment variable with your YouTube Data API key."
    )

youtube = build("youtube", "v3", developerKey=API_KEY)
print("YouTube API connection successful!")


## 3. Get Channel Information

In [ ]:
CHANNEL_HANDLE = "@ApnaCollegeOfficial"

request = youtube.channels().list(
    part="snippet,statistics",
    forHandle=CHANNEL_HANDLE
)
response = request.execute()

if not response.get("items"):
    raise ValueError("Channel not found. Check CHANNEL_HANDLE.")

channel = response["items"][0]

channel_data = {
    "channel_id": channel["id"],
    "channel_name": channel["snippet"]["title"],
    "published_at": channel["snippet"]["publishedAt"],
    "subscriber_count": int(channel["statistics"].get("subscriberCount", 0)),
    "view_count": int(channel["statistics"].get("viewCount", 0)),
    "video_count": int(channel["statistics"].get("videoCount", 0))
}

pd.DataFrame([channel_data])


## 4. Identify the Uploads Playlist

In [ ]:
request = youtube.channels().list(
    part="contentDetails",
    id=channel_data["channel_id"]
)
response = request.execute()

uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print("Uploads playlist ID:", uploads_playlist_id)


## 5. Collect Video IDs Using Pagination

The API is queried in pages of up to 50 playlist items until there is no `nextPageToken`.


In [ ]:
videos = []
next_page_token = None

while True:
    request = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=uploads_playlist_id,
        maxResults=50,
        pageToken=next_page_token
    )
    response = request.execute()
    videos.extend(response["items"])

    next_page_token = response.get("nextPageToken")
    print("Videos collected:", len(videos))

    if not next_page_token:
        break

print("Total videos collected:", len(videos))


In [ ]:
video_ids = [item["contentDetails"]["videoId"] for item in videos]
print("Total video IDs:", len(video_ids))


## 6. Retrieve Detailed Video Information

In [ ]:
video_details = []

for start in range(0, len(video_ids), 50):
    batch_ids = video_ids[start:start + 50]

    request = youtube.videos().list(
        part="snippet,contentDetails,statistics",
        id=",".join(batch_ids)
    )
    response = request.execute()
    video_details.extend(response["items"])

    print("Videos processed:", len(video_details))

print("Total video details:", len(video_details))


## 7. Normalize and Select Required Fields

In [ ]:
df_details = pd.json_normalize(video_details)
print("Raw detailed dataset shape:", df_details.shape)

keep_columns = [
    "id",
    "snippet.publishedAt",
    "snippet.channelTitle",
    "snippet.title",
    "snippet.description",
    "snippet.tags",
    "snippet.categoryId",
    "snippet.defaultLanguage",
    "snippet.defaultAudioLanguage",
    "contentDetails.duration",
    "contentDetails.definition",
    "contentDetails.caption",
    "statistics.viewCount",
    "statistics.likeCount",
    "statistics.commentCount"
]

df = df_details[keep_columns].copy()
print("Selected dataset shape:", df.shape)
df.head()


## 8. Initial Data Quality Checks

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum())

print("Duplicate video IDs:", df["id"].duplicated().sum())
print("Unique video IDs:", df["id"].nunique())


## 9. Clean and Convert Data Types

In [ ]:
df.drop(columns=["snippet.defaultAudioLanguage"], inplace=True)

df["snippet.publishedAt"] = pd.to_datetime(df["snippet.publishedAt"])

for column in [
    "statistics.viewCount",
    "statistics.likeCount",
    "statistics.commentCount"
]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes


## 10. Convert ISO 8601 Duration to Minutes

In [ ]:
def duration_to_minutes(duration):
    match = re.match(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        duration
    )
    if not match:
        return None

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return round(hours * 60 + minutes + seconds / 60, 2)

df["duration_minutes"] = df["contentDetails.duration"].apply(duration_to_minutes)

df[["contentDetails.duration", "duration_minutes"]].head(10)


## 11. Create Engagement Metrics

The project calculates:
- **Like Rate** = Likes / Views × 100
- **Comment Rate** = Comments / Views × 100
- **Engagement Rate** = (Likes + Comments) / Views × 100


In [ ]:
views = df["statistics.viewCount"].replace(0, pd.NA)

df["like_rate"] = (df["statistics.likeCount"] / views) * 100
df["comment_rate"] = (df["statistics.commentCount"] / views) * 100
df["engagement_rate"] = (
    (df["statistics.likeCount"] + df["statistics.commentCount"]) / views
) * 100

df[
    [
        "snippet.title",
        "statistics.viewCount",
        "statistics.likeCount",
        "statistics.commentCount",
        "like_rate",
        "comment_rate",
        "engagement_rate"
    ]
].head(10)


## 12. Create Publishing Date Features

In [ ]:
df["published_year"] = df["snippet.publishedAt"].dt.year
df["published_month"] = df["snippet.publishedAt"].dt.month
df["published_month_name"] = df["snippet.publishedAt"].dt.month_name()
df["published_day"] = df["snippet.publishedAt"].dt.day
df["published_day_name"] = df["snippet.publishedAt"].dt.day_name()
df["published_date"] = df["snippet.publishedAt"].dt.date

df[
    [
        "snippet.publishedAt",
        "published_year",
        "published_month",
        "published_month_name",
        "published_day",
        "published_day_name",
        "published_date"
    ]
].head()


## 13. Data Quality Validation

In [ ]:
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Unique IDs:", df["id"].nunique())
print("Rows:", len(df))

print("\nZero-value checks:")
print("Views = 0:", (df["statistics.viewCount"] == 0).sum())
print("Likes = 0:", (df["statistics.likeCount"] == 0).sum())
print("Comments = 0:", (df["statistics.commentCount"] == 0).sum())

print("\nNegative-value checks:")
print("Negative views:", (df["statistics.viewCount"] < 0).sum())
print("Negative likes:", (df["statistics.likeCount"] < 0).sum())
print("Negative comments:", (df["statistics.commentCount"] < 0).sum())


## 14. Create the SQL-Ready Dataset

In [ ]:
df_sql = df.copy()

df_sql.columns = [
    "id", "published_at", "channel_title", "title", "description", "tags",
    "category_id", "default_language", "duration", "definition",
    "duration_minutes", "view_count", "like_count", "comment_count",
    "like_rate", "comment_rate", "engagement_rate",
    "published_year", "published_month", "published_month_name",
    "published_day", "published_day_name", "published_date",
    "duration_category"
]

# Recreate date features from the standardized timestamp
df_sql["published_year"] = df_sql["published_at"].dt.year
df_sql["published_month"] = df_sql["published_at"].dt.month
df_sql["published_month_name"] = df_sql["published_at"].dt.month_name()
df_sql["published_day"] = df_sql["published_at"].dt.day
df_sql["published_day_name"] = df_sql["published_at"].dt.day_name()
df_sql["published_date"] = df_sql["published_at"].dt.date

# Final duration conversion
df_sql["duration_minutes"] = df_sql["duration"].apply(duration_to_minutes)

# Final duration categories used in the project
def categorize_duration(minutes):
    if minutes <= 10:
        return "Short"
    elif minutes <= 30:
        return "Medium"
    elif minutes <= 60:
        return "Long"
    else:
        return "Very Long"

df_sql["duration_category"] = df_sql["duration_minutes"].apply(categorize_duration)

# Final rounding
df_sql["duration_minutes"] = df_sql["duration_minutes"].round(2)
df_sql["like_rate"] = df_sql["like_rate"].round(2)
df_sql["comment_rate"] = df_sql["comment_rate"].round(2)
df_sql["engagement_rate"] = df_sql["engagement_rate"].round(2)

df_sql.head()


## 15. Final Dataset Validation

In [ ]:
print("Final dataset shape:", df_sql.shape)
print("Number of columns:", len(df_sql.columns))
print("\nColumns:")
print(df_sql.columns.tolist())

print("\nMissing values:")
display(df_sql.isnull().sum())

display(df_sql.head())


## 16. Exploratory Analysis

In [ ]:
analysis_columns = [
    "view_count",
    "like_count",
    "comment_count",
    "duration_minutes",
    "engagement_rate"
]

df_sql[analysis_columns].describe()


In [ ]:
print("Median values:")
display(df_sql[analysis_columns].median())


## 17. Top 10 Videos by Views

In [ ]:
top_10_views = df_sql.sort_values("view_count", ascending=False).head(10)

display(
    top_10_views[
        [
            "title", "view_count", "like_count",
            "comment_count", "duration_minutes", "engagement_rate"
        ]
    ].reset_index(drop=True)
)


## 18. Top 10 Most Liked Videos

In [ ]:
top_10_likes = df_sql.sort_values("like_count", ascending=False).head(10)

display(
    top_10_likes[
        ["title", "view_count", "like_count", "comment_count", "engagement_rate"]
    ].reset_index(drop=True)
)


## 19. Top 10 Most Commented Videos

In [ ]:
top_10_comments = df_sql.sort_values("comment_count", ascending=False).head(10)

display(
    top_10_comments[
        ["title", "view_count", "like_count", "comment_count", "engagement_rate"]
    ].reset_index(drop=True)
)


## 20. Top 10 Videos by Engagement Rate

In [ ]:
top_10_engagement = df_sql.sort_values("engagement_rate", ascending=False).head(10)

display(
    top_10_engagement[
        ["title", "view_count", "like_count", "comment_count", "engagement_rate"]
    ].reset_index(drop=True)
)


## 21. Duration Analysis

In [ ]:
duration_analysis = (
    df_sql
    .groupby("duration_category", observed=True)
    .agg(
        video_count=("id", "count"),
        average_views=("view_count", "mean"),
        median_views=("view_count", "median"),
        average_engagement=("engagement_rate", "mean")
    )
    .reset_index()
)

display(duration_analysis)


## 22. Year-wise Analysis

In [ ]:
year_analysis = (
    df_sql
    .groupby("published_year")
    .agg(
        video_count=("id", "count"),
        average_views=("view_count", "mean"),
        median_views=("view_count", "median"),
        average_engagement=("engagement_rate", "mean")
    )
    .reset_index()
)

display(year_analysis)
display(year_analysis.sort_values("average_views", ascending=False))


## 23. Month-wise Analysis

In [ ]:
month_analysis = (
    df_sql
    .groupby(["published_month", "published_month_name"])
    .agg(
        video_count=("id", "count"),
        average_views=("view_count", "mean"),
        average_engagement=("engagement_rate", "mean")
    )
    .reset_index()
    .sort_values("published_month")
)

display(month_analysis)
display(month_analysis.sort_values("average_views", ascending=False))


## 24. Category Analysis

In [ ]:
category_analysis = (
    df_sql
    .groupby("category_id")
    .agg(
        video_count=("id", "count"),
        average_views=("view_count", "mean"),
        average_engagement=("engagement_rate", "mean")
    )
    .reset_index()
)

display(category_analysis.sort_values("average_views", ascending=False))


## 25. Export Cleaned Datasets

In [ ]:
df.to_csv("youtube_cleaned_data.csv", index=False)
df_sql.to_csv("youtube_cleaned_sql.csv", index=False)

print("Saved:")
print("- youtube_cleaned_data.csv")
print("- youtube_cleaned_sql.csv")


## 26. Final Output

The SQL-ready dataset contains video metadata, publishing features, duration information, performance metrics, engagement metrics, and duration categories.

The file `youtube_cleaned_sql.csv` is the dataset intended for the MySQL analysis script and downstream Power BI work.

### GitHub Security Checklist
- API keys are not stored in the notebook.
- Database passwords are not stored in the notebook.
- Personal Windows file paths are not required.
- Generated CSV files can be placed in the repository's `data/` folder.
